In [1]:
from datasets import load_dataset
import pandas as pd
from openai import OpenAI
import os
import json
import numpy as np

c:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Filter Hanoi 
Only take province named Hanoi.

In [2]:
df = load_dataset("tinixai/vietnam-real-estates")
df = df["train"].to_pandas()
hanoi = df.loc[df["province_name"].isin(["Hà Nội"])]

# Create QA test set

In [3]:
hanoi['Billion VND'] = hanoi['price'] / 1e9
hanoi = hanoi[['name', 'property_type_name', 'district_name', 'bedroom_count', 'Billion VND', 'area']]
hanoi.dropna(axis=0, inplace=True)
display(hanoi.head())
display(hanoi.shape)

C:\Users\s223128143\AppData\Local\Temp\ipykernel_34968\3207283414.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hanoi['Billion VND'] = hanoi['price'] / 1e9


,name,property_type_name,district_name,bedroom_count,Billion VND,area
0,"Bán nhanh nhà 6 tầng thang máy, ngoạ long cầu ...",Nhà,Bắc Từ Liêm,4.0,7.45,37.0
11,Bán nhanh căn góc 3 ngủ/90m2 Sunshine Riversid...,Căn hộ chung cư,Tây Hồ,3.0,7.40,90.0
17,"Phân lô, ô tô, vỉa hè - Phố Hoàng Quốc Việt, C...",Nhà,Cầu Giấy,4.0,16.70,41.0
24,Bán nhà Hoàng Hoa Thám - Ngọc Hà - Ba Đình - 3...,Nhà,Ba Đình,3.0,5.35,32.0
29,Chính chủ bán căn hộ 66m² – 2 phòng ngủ full n...,Căn hộ chung cư,Thanh Oai,2.0,2.64,66.0


(616186, 6)

In [4]:
# Take 5000 listing from hanoi
hanoi = hanoi.head(5000)
hanoi.shape

(5000, 6)

In [5]:
hanoi.to_csv("hanoi.csv", index=False)

In [ ]:
from openai import OpenAI
import os
import json

client = OpenAI(api_key="")
model = "gpt-5.4-nano"
system_prompt = """
You are helping create a QA test set for a Vietnamese real estate RAG evaluation.

Given a property listing, generate exactly 3 questions that all mean the same thing but use different styles.
Each question must be answerable by this single listing and no other.

Rules:
1. CLEAN — Full Vietnamese, no abbreviations, no English.
   - Write out "phòng ngủ", "quận", "tỷ đồng", "mét vuông" in full.
   - Use a price ceiling of actual price + 0.5 tỷ.

2. ADDRESS VARIANT — Vietnamese with real estate shorthand.
   - "phòng ngủ" → "PN", "quận" → "Q", "tỷ" stays as "tỷ".

3. CODE-SWITCHED — Mix of English and Vietnamese abbreviations.
   - Property type in English: "apartment", "house", "land".
   - "tỷ" → "B", "phòng ngủ" → "BR".
   - District name stays in Vietnamese.

Additional rules:
- Always use a price ceiling (under X), not an exact price.
- Output ONLY valid JSON, no extra text:
{
  "listing_id": "...",
  "clean": "...",
  "address_variant": "...",
  "code_switched": "..."
}
- The questions must read like a real buyer search query with specific values filled in.
Do NOT use "bao nhiêu" (how many) — state the actual numbers from the listing.
Do NOT ask open-ended questions. Each question should be a specific search query
that a buyer would type to find exactly this property.

Do NOT output multiple JSON objects. Do NOT wrap in a list.
"""

def listing_prompt(row):
    return f"""Listing:
    - listing_id: {row.name}
    - property_type_name: {row['property_type_name']}
    - district_name: {row['district_name']}
    - bedroom_count: {row['bedroom_count']}
    - Billion_VND: {row['Billion VND']}
    - area: {row['area']}"""

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def QnA_dataset(row):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": system_prompt},
                {"role": "user", "content": listing_prompt(row)},
            ],
            temperature=0,
            response_format={"type": "json_object"},
        )
        result = json.loads(response.choices[0].message.content)
        return result
    except Exception as e:
        print(f"Error on listing {row.name}: {e}")
        return None

MAX_WORKERS = 10

rows = [row for _, row in hanoi.iterrows()]

qa_dataset = [None] * len(rows)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(QnA_dataset, row): i for i, row in enumerate(rows)}
    for future in tqdm(as_completed(futures), total=len(rows)):
        i = futures[future]
        qa_dataset[i] = future.result()

qa_dataset = [q for q in qa_dataset if q is not None]  # drop failed rows
qa_df = pd.DataFrame(qa_dataset)
qa_df.to_csv("qa_dataset.csv", index=False)

100%|██████████| 5000/5000 [14:53<00:00,  5.60it/s]


In [2]:
from sentence_transformers import SentenceTransformer, models
from sklearn.metrics.pairwise import cosine_similarity

# Corpus encoding

In [3]:
qa_df = pd.read_csv("qa_dataset.csv")
hanoi = pd.read_csv("hanoi.csv")

In [4]:
phobert_transformer = models.Transformer(
    "vinai/phobert-base",
    max_seq_length=256,
)
phobert_pooling = models.Pooling(
    phobert_transformer.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True,
)
phobert = SentenceTransformer(modules=[phobert_transformer, phobert_pooling])

In [5]:
vibe = SentenceTransformer("bkai-foundation-models/vietnamese-bi-encoder")

In [6]:
me5 = SentenceTransformer("intfloat/multilingual-e5-base")

In [7]:
# Build document corpus
def build_document(row):
    parts = [
        str(row['property_type_name']),
        f"quận {row['district_name']}",
        f"{row['bedroom_count']:.0f} phòng ngủ" if pd.notna(row['bedroom_count']) else "",
        f"diện tích {row['area']} mét vuông" if pd.notna(row['area']) else "",
        f"giá {row['Billion VND']:.2f} tỷ đồng",
    ]
    return " ".join(p for p in parts if p)

hanoi['document'] = hanoi.apply(build_document, axis=1)

In [8]:
docs = hanoi["document"].tolist()

model_dict = {
    "PhoBERT-base": phobert,
    "vietnamese-bi-encoder": vibe,
    "multilingual-e5-base": me5,
}

variants = ["clean", "address_variant", "code_switched"]

def encode_corpus(model, docs, is_e5=False):
    texts = [f"passage: {d}" if is_e5 else d for d in docs]
    return model.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=True)

def encode_queries(model, qa_df, is_e5=False):
    out = {}
    for v in variants:
        texts = [f"query: {q}" if is_e5 else q for q in qa_df[v].tolist()]
        out[v] = model.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=False)
    return out

encodings = {}
for name, model in model_dict.items():
    is_e5 = "e5" in name.lower()
    print(f"\nEncoding with {name}...")
    encodings[name] = {
        "docs": encode_corpus(model, docs, is_e5),
        "queries": encode_queries(model, qa_df, is_e5),
    }


Encoding with PhoBERT-base...


Batches: 100%|██████████| 157/157 [00:02<00:00, 56.30it/s]



Encoding with vietnamese-bi-encoder...


Batches: 100%|██████████| 157/157 [00:02<00:00, 67.49it/s]



Encoding with multilingual-e5-base...


Batches: 100%|██████████| 157/157 [00:02<00:00, 73.28it/s]


# Retrieval and Evaluation

In [9]:
def compute_metrics(doc_embs, query_embs, k=10):
    sim = cosine_similarity(query_embs, doc_embs)
    n = len(query_embs)
    recall_at_1 = np.zeros(n)
    mrr_scores  = np.zeros(n)
    for i in range(n):
        ranked = np.argsort(sim[i])[::-1]
        recall_at_1[i] = 1.0 if ranked[0] == i else 0.0
        pos = np.where(ranked[:k] == i)[0]
        if len(pos) > 0:
            mrr_scores[i] = 1.0 / (pos[0] + 1)
    return recall_at_1, mrr_scores

In [10]:
rows = []
mrr_raw = {}

for model_name, enc in encodings.items():
    mrr_raw[model_name] = {}
    for v in variants:
        r1, mrr = compute_metrics(enc["docs"], enc["queries"][v])
        mrr_raw[model_name][v] = mrr
        rows.append({
            "Model": model_name,
            "Query Type": v,
            "Recall@1": round(r1.mean(), 4),
            "MRR@10": round(mrr.mean(), 4),
            "n": len(r1),
        })

results_df = pd.DataFrame(rows)

In [11]:
variant_order = ["clean", "address_variant", "code_switched"]
results_df["Query Type"] = pd.Categorical(results_df["Query Type"], categories=variant_order, ordered=True)

pivot = results_df.sort_values(["Model", "Query Type"]).pivot_table(index="Model", columns="Query Type", values=["Recall@1", "MRR@10"]).reindex(variant_order, axis=1, level=1)

display(pivot)

C:\Users\s223128143\AppData\Local\Temp\ipykernel_25624\2527887123.py:4: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = results_df.sort_values(["Model", "Query Type"]).pivot_table(index="Model", columns="Query Type", values=["Recall@1", "MRR@10"]).reindex(variant_order, axis=1, level=1)


MRR@10                               Recall@1  \
Query Type              clean address_variant code_switched    clean   
Model                                                                  
PhoBERT-base           0.0631          0.0523        0.0168   0.0352   
multilingual-e5-base   0.4976          0.4834        0.4692   0.3836   
vietnamese-bi-encoder  0.2932          0.2292        0.1370   0.1878   

                                                     
Query Type            address_variant code_switched  
Model                                                
PhoBERT-base                   0.0304        0.0072  
multilingual-e5-base           0.3664        0.3480  
vietnamese-bi-encoder          0.1388        0.0764

# Error analysis

In [25]:
vocab_probes = [
    ("phòng ngủ", "PN"),         
    ("căn hộ", "apartment"),    
    ("tỷ đồng", "B"),           
    ("quận", "Q"),  
    ("nhà", "house"),   
]

vibe = model_dict["vietnamese-bi-encoder"]
me5 = model_dict["multilingual-e5-base"]

probe_rows = []
for (full, code_switch) in vocab_probes:
    v_embs = vibe.encode([full, code_switch], normalize_embeddings=True)
    e_embs = me5.encode([f"query: {full}", f"query: {code_switch}"], normalize_embeddings=True)
    vibe_sim  = float(cosine_similarity([v_embs[0]], [v_embs[1]])[0, 0])
    me5_sim  = float(cosine_similarity([e_embs[0]], [e_embs[1]])[0, 0])
    probe_rows.append({"clean": full, "code_switch": code_switch, "vibe_sim": vibe_sim, "me5_sim": me5_sim})

probe_df = pd.DataFrame(probe_rows)

In [26]:
print(probe_df)

       clean code_switch  vibe_sim   me5_sim
0  phòng ngủ          PN  0.115918  0.795491
1     căn hộ   apartment  0.334001  0.887792
2    tỷ đồng           B  0.034678  0.798910
3       quận           Q  0.154040  0.813944
4        nhà       house  0.724291  0.923619
